In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import RFE, chi2, SelectKBest
from sklearn.linear_model import LogisticRegression
df=pd.read_csv('../data/heart.csv')
y=df['target']
x=df.drop(columns=['target'])
num=x.select_dtypes(include=[int,float]).columns.tolist()
cat=[c for c in x.columns if c not in num]
pre_num=Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())])
pre_cat=Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))])
pre=ColumnTransformer([('num',pre_num,num),('cat',pre_cat,cat)])
est=LogisticRegression(max_iter=1000,class_weight='balanced')
pipe=Pipeline([('pre',pre),('clf',est)]).fit(x,y)
rfe=RFE(estimator=LogisticRegression(max_iter=1000,class_weight='balanced'),n_features_to_select=8)
rfe=rfe.fit(pre.fit_transform(x),y)
oh=pipe.named_steps['pre'].named_transformers_['cat'].named_steps['oh'] if len(cat)>0 else None
oh_names=list(oh.get_feature_names_out(cat)) if oh is not None else []
feat=num+oh_names
import pandas as pd
pd.DataFrame({'feature':feat,'selected':rfe.get_support()}).to_csv('../results/feature_rfe.csv',index=False)
from sklearn.preprocessing import MinMaxScaler
numX=df[num].fillna(df[num].median())
chiX=MinMaxScaler().fit_transform(numX)
from sklearn.feature_selection import SelectKBest
sel=SelectKBest(chi2,k=min(8,chiX.shape[1])).fit(chiX,y)
pd.DataFrame({'feature':num,'selected':sel.get_support()}).to_csv('../results/feature_chi2.csv',index=False)